# 🧪 Test the Validation System

This notebook lets you test the validation system **without any API keys or external services**.

You can run this locally to see how the validation, progress tracking, and achievements work!

## Load the Validation System

In [ ]:
# Load validation system (no external dependencies)
import inspect
import os
import time
from typing import Any, Tuple, List, Dict, Callable
from datetime import datetime
from IPython.display import display, HTML
import ipywidgets as widgets

class ValidationResult:
    """Represents the result of a validation check"""
    def __init__(self, passed: bool, message: str, hints: List[str] = None, details: Dict = None):
        self.passed = passed
        self.message = message
        self.hints = hints or []
        self.details = details or {}
        self.timestamp = datetime.now()
    
    def display(self):
        """Display validation result with formatting"""
        if self.passed:
            status_html = '<div style="padding: 10px; background-color: #d4edda; border: 1px solid #c3e6cb; border-radius: 5px; margin: 10px 0;">'
            status_html += '<h3 style="color: #155724;">✅ Validation Passed!</h3>'
            status_html += f'<p style="color: #155724;">{self.message}</p>'
        else:
            status_html = '<div style="padding: 10px; background-color: #f8d7da; border: 1px solid #f5c6cb; border-radius: 5px; margin: 10px 0;">'
            status_html += '<h3 style="color: #721c24;">❌ Validation Failed</h3>'
            status_html += f'<p style="color: #721c24;">{self.message}</p>'
            
            if self.hints:
                status_html += '<h4 style="color: #856404;">💡 Hints:</h4>'
                status_html += '<ul style="color: #856404;">'
                for hint in self.hints:
                    status_html += f'<li>{hint}</li>'
                status_html += '</ul>'
        
        status_html += '</div>'
        display(HTML(status_html))

class LabProgress:
    """Tracks student progress through the lab"""
    def __init__(self):
        self.completed_blocks = set()
        self.attempts = {}
        self.start_time = datetime.now()
    
    def record_attempt(self, block_num: int, passed: bool):
        if block_num not in self.attempts:
            self.attempts[block_num] = []
        self.attempts[block_num].append({
            'passed': passed,
            'timestamp': datetime.now()
        })
        if passed:
            self.completed_blocks.add(block_num)
            # Check achievements
            achievements.check_achievements(self.completed_blocks)
    
    def display_progress(self, total_blocks: int = 5):  # Using 5 test blocks
        percentage = (len(self.completed_blocks) / total_blocks) * 100
        completed = len(self.completed_blocks)
        
        progress_html = f'''
        <div style="margin: 20px 0;">
            <h3>📊 Your Progress: {completed}/{total_blocks} CODE_BLOCKs completed ({percentage:.1f}%)</h3>
            <div style="width: 100%; background-color: #e0e0e0; border-radius: 10px; overflow: hidden;">
                <div style="width: {percentage}%; background-color: #4CAF50; height: 30px; text-align: center; line-height: 30px; color: white;">
                    {percentage:.1f}%
                </div>
            </div>
        </div>
        '''
        display(HTML(progress_html))

class AchievementSystem:
    """Gamification through achievements"""
    
    def __init__(self):
        self.achievements = {
            'first_step': {'name': '🎉 First Steps', 'desc': 'Complete your first CODE_BLOCK', 'blocks_needed': 1},
            'halfway_hero': {'name': '⭐ Halfway Hero', 'desc': 'Complete 3 CODE_BLOCKs', 'blocks_needed': 3},
            'test_champion': {'name': '🏆 Test Champion', 'desc': 'Complete all test CODE_BLOCKs!', 'blocks_needed': [1, 2, 3, 4, 5]}
        }
        self.earned = set()
    
    def check_achievements(self, completed_blocks: set):
        """Check for newly earned achievements"""
        newly_earned = []
        
        for ach_id, ach_data in self.achievements.items():
            if ach_id not in self.earned:
                blocks_needed = ach_data['blocks_needed']
                
                if isinstance(blocks_needed, int):
                    if len(completed_blocks) >= blocks_needed:
                        self.earned.add(ach_id)
                        newly_earned.append(ach_data)
                else:
                    if all(b in completed_blocks for b in blocks_needed):
                        self.earned.add(ach_id)
                        newly_earned.append(ach_data)
        
        for achievement in newly_earned:
            self._display_achievement(achievement)
    
    def _display_achievement(self, achievement: dict):
        """Display achievement notification"""
        html = f"""
        <div style="
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            color: white;
            padding: 20px;
            border-radius: 10px;
            margin: 20px 0;
            text-align: center;
            box-shadow: 0 4px 6px rgba(0,0,0,0.1);
        ">
            <h2 style="margin: 0; font-size: 48px;">🎊</h2>
            <h3 style="margin: 10px 0;">Achievement Unlocked!</h3>
            <h2 style="margin: 10px 0;">{achievement['name']}</h2>
            <p style="margin: 5px 0; opacity: 0.9;">{achievement['desc']}</p>
        </div>
        """
        display(HTML(html))

# Global instances
lab_progress = LabProgress()
achievements = AchievementSystem()

def create_validation_widget(block_num: int, validator_func: Callable, variable_name: str):
    """Creates an interactive validation widget for a CODE_BLOCK"""
    
    button = widgets.Button(
        description=f'Validate CODE_BLOCK_{block_num}',
        button_style='primary',
        icon='check'
    )
    
    output = widgets.Output()
    
    def on_click(b):
        with output:
            output.clear_output()
            
            try:
                var = globals().get(variable_name)
                if var is None:
                    result = ValidationResult(
                        False,
                        f"Variable '{variable_name}' not found.",
                        hints=[f"Make sure you've run the CODE_BLOCK_{block_num} cell and created '{variable_name}'"]
                    )
                else:
                    result = validator_func(var)
                    lab_progress.record_attempt(block_num, result.passed)
                
                result.display()
                
                if result.passed:
                    lab_progress.display_progress()
                    
            except Exception as e:
                result = ValidationResult(
                    False,
                    f"Error during validation: {str(e)}",
                    hints=["Check your code for syntax errors"]
                )
                result.display()
    
    button.on_click(on_click)
    
    return widgets.VBox([button, output])

print("✅ Validation system loaded successfully!")
lab_progress.display_progress()

## Test Exercise 1: Simple Variable

Create a variable called `my_name` with your name as a string.

In [ ]:
# CODE_BLOCK_1: Create a variable called 'my_name'
# Try different values to see validation in action:
# 1. First try: my_name = 123  (should fail - not a string)
# 2. Then try: my_name = ""  (should fail - empty string)
# 3. Finally: my_name = "Your Name"  (should pass)

# <CODE_BLOCK_1>
# my_name = "Your Name Here"

In [ ]:
# Validation for CODE_BLOCK_1
def validate_test_block_1(name_var):
    if not isinstance(name_var, str):
        return ValidationResult(
            False,
            "The variable should be a string.",
            hints=["Use quotes around your name", "Example: my_name = \"John Doe\""]
        )
    
    if len(name_var) == 0:
        return ValidationResult(
            False,
            "The name shouldn't be empty.",
            hints=["Add your actual name to the string"]
        )
    
    return ValidationResult(
        True,
        f"Great job, {name_var}! You've completed your first CODE_BLOCK! 🎉"
    )

display(create_validation_widget(1, validate_test_block_1, 'my_name'))

## Test Exercise 2: Create a List

Create a list called `numbers` with at least 3 numbers.

In [ ]:
# CODE_BLOCK_2: Create a list of numbers
# Try these to test validation:
# 1. numbers = "not a list"  (should fail)
# 2. numbers = [1, 2]  (should fail - not enough items)
# 3. numbers = [1, 2, 3, 4, 5]  (should pass)

# <CODE_BLOCK_2>
# numbers = [1, 2, 3, 4, 5]

In [ ]:
# Validation for CODE_BLOCK_2
def validate_test_block_2(numbers_var):
    if not isinstance(numbers_var, list):
        return ValidationResult(
            False,
            "The variable should be a list.",
            hints=["Use square brackets: [1, 2, 3]"]
        )
    
    if len(numbers_var) < 3:
        return ValidationResult(
            False,
            f"List has {len(numbers_var)} items, but needs at least 3.",
            hints=["Add more numbers to your list"]
        )
    
    return ValidationResult(
        True,
        f"Excellent! Your list has {len(numbers_var)} numbers. 📊"
    )

display(create_validation_widget(2, validate_test_block_2, 'numbers'))

## Test Exercise 3: Define a Function

Create a function called `double_number` that takes a number and returns it doubled.

In [ ]:
# CODE_BLOCK_3: Define the function
# Test different implementations:
# 1. double_number = "not a function"  (should fail)
# 2. def double_number(x): pass  (should fail - doesn't return anything)
# 3. def double_number(x): return x * 2  (should pass)

# <CODE_BLOCK_3>
# def double_number(x):
#     return x * 2

In [ ]:
# Validation for CODE_BLOCK_3
def validate_test_block_3(func):
    if not callable(func):
        return ValidationResult(
            False,
            "double_number should be a function.",
            hints=["Use 'def double_number(x):' to define it"]
        )
    
    # Test the function
    try:
        result = func(5)
        if result != 10:
            return ValidationResult(
                False,
                f"Function returned {result} instead of 10 for input 5.",
                hints=["Return x * 2", "Make sure to use the return statement"]
            )
        
        return ValidationResult(
            True,
            "Perfect! Your function works correctly. 🎯"
        )
    except Exception as e:
        return ValidationResult(
            False,
            f"Function error: {str(e)}",
            hints=["Check your function implementation"]
        )

display(create_validation_widget(3, validate_test_block_3, 'double_number'))

## Test Exercise 4: Work with Dictionaries

Create a dictionary called `person` with keys 'name' and 'age'.

In [ ]:
# CODE_BLOCK_4: Create a dictionary
# Test scenarios:
# 1. person = []  (should fail - not a dict)
# 2. person = {"name": "John"}  (should fail - missing 'age')
# 3. person = {"name": "John", "age": 25}  (should pass)

# <CODE_BLOCK_4>
# person = {"name": "John", "age": 25}

In [ ]:
# Validation for CODE_BLOCK_4
def validate_test_block_4(person_var):
    if not isinstance(person_var, dict):
        return ValidationResult(
            False,
            "person should be a dictionary.",
            hints=["Use curly braces: {\"key\": \"value\"}"]
        )
    
    required_keys = ['name', 'age']
    missing_keys = [key for key in required_keys if key not in person_var]
    
    if missing_keys:
        return ValidationResult(
            False,
            f"Dictionary is missing keys: {missing_keys}",
            hints=["Include both 'name' and 'age' keys"]
        )
    
    return ValidationResult(
        True,
        f"Great! Person dictionary created with name '{person_var['name']}' and age {person_var['age']}. 👤"
    )

display(create_validation_widget(4, validate_test_block_4, 'person'))

## Test Exercise 5: Complex Validation

Create a list called `students` containing at least 2 student dictionaries, each with 'name' and 'grade' keys.

In [ ]:
# CODE_BLOCK_5: Create a complex data structure
# Test scenarios:
# 1. students = "not a list"  (should fail)
# 2. students = [{"name": "Alice"}]  (should fail - missing grade, not enough students)
# 3. students = [{"name": "Alice", "grade": 95}, {"name": "Bob", "grade": 87}]  (should pass)

# <CODE_BLOCK_5>
# students = [
#     {"name": "Alice", "grade": 95},
#     {"name": "Bob", "grade": 87}
# ]

In [ ]:
# Validation for CODE_BLOCK_5
def validate_test_block_5(students_var):
    if not isinstance(students_var, list):
        return ValidationResult(
            False,
            "students should be a list.",
            hints=["Use square brackets to create a list"]
        )
    
    if len(students_var) < 2:
        return ValidationResult(
            False,
            f"Need at least 2 students, found {len(students_var)}.",
            hints=["Add more student dictionaries to the list"]
        )
    
    for i, student in enumerate(students_var):
        if not isinstance(student, dict):
            return ValidationResult(
                False,
                f"Student {i+1} is not a dictionary.",
                hints=["Each student should be a dictionary with name and grade"]
            )
        
        required_keys = ['name', 'grade']
        missing_keys = [key for key in required_keys if key not in student]
        
        if missing_keys:
            return ValidationResult(
                False,
                f"Student {i+1} is missing keys: {missing_keys}",
                hints=["Each student needs both 'name' and 'grade' keys"]
            )
    
    avg_grade = sum(s['grade'] for s in students_var) / len(students_var)
    return ValidationResult(
        True,
        f"🎓 Excellent! Created {len(students_var)} students with average grade {avg_grade:.1f}"
    )

display(create_validation_widget(5, validate_test_block_5, 'students'))

## 🎊 Test Completion Check

In [ ]:
# Check final progress
lab_progress.display_progress()

if len(lab_progress.completed_blocks) == 5:
    display(HTML("""
    <div style="
        background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
        color: white;
        padding: 30px;
        border-radius: 15px;
        text-align: center;
        margin: 20px 0;
        box-shadow: 0 4px 6px rgba(0,0,0,0.1);
    ">
        <h1 style="margin: 0; font-size: 48px;">🏆</h1>
        <h2 style="margin: 10px 0;">Test Champion!</h2>
        <p style="margin: 10px 0; font-size: 18px;">You've completed all validation tests!</p>
        <p style="margin: 5px 0; opacity: 0.9;">The validation system is working perfectly! 🚀</p>
        <div style="background: rgba(255,255,255,0.2); padding: 15px; border-radius: 10px; margin: 15px 0;">
            <h4>✅ Features Tested:</h4>
            <ul style="text-align: left; display: inline-block;">
                <li>Real-time validation feedback</li>
                <li>Progress tracking</li>
                <li>Achievement notifications</li>
                <li>Helpful error hints</li>
                <li>Interactive widgets</li>
            </ul>
        </div>
    </div>
    """))
else:
    remaining = 5 - len(lab_progress.completed_blocks)
    display(HTML(f"""
    <div style="
        background-color: #fff3cd;
        border: 1px solid #ffeaa7;
        color: #856404;
        padding: 20px;
        border-radius: 10px;
        margin: 20px 0;
    ">
        <h3>Keep testing! 💪</h3>
        <p>You have {remaining} more exercise(s) to test.</p>
        <p>Try implementing each CODE_BLOCK and clicking the validation buttons!</p>
    </div>
    """))

## 🧪 Testing Instructions

### How to Test:

1. **For each exercise above**:
   - First, try the **wrong** implementation (as suggested in comments)
   - Click the validation button to see the failure message and hints
   - Then implement the **correct** solution
   - Click validate again to see success and progress update

2. **Watch for**:
   - ✅ Green success messages
   - ❌ Red failure messages with specific hints
   - 📊 Progress bar updates
   - 🎊 Achievement notifications

3. **Try edge cases**:
   - Empty values
   - Wrong types
   - Missing required fields
   - Boundary conditions

### What This Demonstrates:
- The validation system works **without any external APIs**
- Students get **immediate feedback** on their code
- **Progress tracking** keeps them motivated
- **Achievements** make it fun and engaging
- **Specific hints** guide them to the solution

This same system is integrated into the full lab (`new_lab.ipynb`) with all 19 CODE_BLOCKs!